# Demonstração do modelo final — inferência histórica

Este notebook carrega o pipeline final da Sprint 5 e demonstra como obter a probabilidade de IQAr > 100 na hora seguinte e aplicar o limiar congelado de 0,3. Ele usa somente features do período de desenvolvimento, anteriores ao início do holdout, e não lê os rótulos.

> **Importante:** esta é uma demonstração do caminho de inferência, não uma nova avaliação. O artefato foi treinado com o período usado aqui, portanto a amostra é *in-sample*. O resultado não estima desempenho, não é uma previsão atual nem um alerta oficial. O holdout não é carregado, reavaliado ou usado para ajuste.

Execute as células na ordem a partir da raiz do repositório. O arquivo `models/modelo_final_sprint5.joblib` e o CSV integrado/rotulado do período configurado precisam existir.

In [1]:
from pathlib import Path
import sys

import joblib
import pandas as pd
import yaml
from IPython.display import display

ROOT = next(
    (pasta for pasta in (Path.cwd(), *Path.cwd().parents)
     if (pasta / "config" / "params.yaml").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Abra o notebook dentro do repositório, que deve conter config/params.yaml."
    )
sys.path.insert(0, str(ROOT))

from src.transformacao.Features_Temporais import adicionar_resumos_curto_prazo

print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: c:\Projetos\Ciencia-de-Dados-e-Aprendizagem-de-Maquina


## 1. Carregar a configuração e o artefato

O artefato guarda o pipeline já treinado, as features esperadas e o limiar escolhido na validação. Nenhum treinamento acontece neste notebook.

In [2]:
with (ROOT / "config" / "params.yaml").open("r", encoding="utf-8") as arquivo:
    config = yaml.safe_load(arquivo)

caminho_modelo = ROOT / "models" / "modelo_final_sprint5.joblib"
periodo = {
    "start_date": config["collection"]["start_date"],
    "end_date": config["collection"]["end_date"],
}
caminho_dados = (
    ROOT
    / config["paths"]["interim_data"]
    / config["paths"]["labeled_filename"].format(**periodo)
)

for caminho in (caminho_modelo, caminho_dados):
    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo necessário não encontrado: {caminho}")

artefato = joblib.load(caminho_modelo)
chaves_necessarias = {"pipeline", "features", "decision_threshold"}
if not isinstance(artefato, dict) or not chaves_necessarias.issubset(artefato):
    raise ValueError(
        "O artefato não contém pipeline, lista de features e limiar esperados."
    )

pipeline = artefato["pipeline"]
features = list(artefato["features"])
limiar = float(artefato["decision_threshold"])
if not 0 <= limiar <= 1:
    raise ValueError(f"Limiar inválido no artefato: {limiar}")

print(f"Modelo: {artefato['model_name']}")
print(f"Limiar congelado na validação: {limiar:.1f}")
print(f"Número de features: {len(features)}")

Modelo: random_forest_balanced
Limiar congelado na validação: 0.3
Número de features: 15


## 2. Ler somente o desenvolvimento

A coluna do alvo não é lida. O número de linhas é calculado pelo início do teste, pelo horizonte e pela regra `event_time < início do teste`; assim, a leitura termina antes do holdout.

In [3]:
inicio_teste = pd.Timestamp(config["evaluation"]["final_test"]["start"])
horizonte = pd.Timedelta(
    hours=int(config["evaluation"]["target_horizon_hours"])
)
limite_exclusivo_time = inicio_teste - horizonte

colunas_temporais = (
    "media_3h_ozone",
    "media_3h_pm2_5",
    "delta_2h_ozone",
    "delta_2h_pm2_5",
)
colunas_base = [coluna for coluna in features if coluna not in colunas_temporais]
colunas_leitura = list(dict.fromkeys([
    "time",
    *colunas_base,
    "ozone",
    "pm2_5",
]))

cabecalho = pd.read_csv(caminho_dados, nrows=0).columns
colunas_ausentes = sorted(set(colunas_leitura).difference(cabecalho))
if colunas_ausentes:
    raise ValueError(f"Colunas necessárias ausentes no CSV: {colunas_ausentes}")

primeira_linha = pd.read_csv(
    caminho_dados,
    usecols=["time"],
    parse_dates=["time"],
    nrows=1,
)
if primeira_linha.empty or pd.isna(primeira_linha["time"].iloc[0]):
    raise ValueError("Não foi possível identificar o primeiro horário do CSV.")
primeiro_horario = primeira_linha["time"].iloc[0]
horas_ate_corte = (limite_exclusivo_time - primeiro_horario).total_seconds() / 3600
if horas_ate_corte <= 0 or not horas_ate_corte.is_integer():
    raise ValueError(
        "A origem do CSV e o corte do holdout não permitem calcular um prefixo horário exato."
    )
quantidade_linhas = int(horas_ate_corte)

dados = pd.read_csv(
    caminho_dados,
    usecols=colunas_leitura,
    parse_dates=["time"],
    nrows=quantidade_linhas,
)
if dados.empty or dados["time"].isna().any():
    raise ValueError("O prefixo de desenvolvimento está vazio ou contém horário ausente.")
if dados["time"].duplicated().any() or not dados["time"].is_monotonic_increasing:
    raise ValueError("Os horários de entrada devem ser únicos e cronológicos.")
if not dados["time"].diff().iloc[1:].eq(pd.Timedelta(hours=1)).all():
    raise ValueError("O prefixo lido não é uma série horária contínua.")

eventos_desenvolvimento = dados["time"] + horizonte
if not eventos_desenvolvimento.lt(inicio_teste).all():
    raise RuntimeError("A leitura incluiu horário pertencente ao holdout.")

print(f"Linhas lidas: {len(dados):,}")
print(f"Último horário de entrada: {dados['time'].iloc[-1]}")
print(f"Último evento previsto: {eventos_desenvolvimento.iloc[-1]}")
print(f"Início do holdout (não lido): {inicio_teste}")

Linhas lidas: 21,143
Último horário de entrada: 2024-12-31 22:00:00
Último evento previsto: 2024-12-31 23:00:00
Início do holdout (não lido): 2025-01-01 00:00:00


## 3. Construir as features e escolher uma amostra ilustrativa

As médias usam três medições até *t*; os deltas comparam *t* e *t−2h*. A amostra escolhida é a última linha do desenvolvimento com esses atributos disponíveis. Não se consulta nem se exibe seu rótulo real.

In [4]:
dados_com_features = adicionar_resumos_curto_prazo(
    dados,
    ("ozone", "pm2_5"),
)
candidatas = dados_com_features.loc[
    dados_com_features[list(colunas_temporais)].notna().all(axis=1)
]
if candidatas.empty:
    raise ValueError("Não há linha com histórico suficiente para calcular as features.")

amostra = candidatas.iloc[[-1]]
entrada_modelo = amostra[features]
if not (amostra["time"].iloc[0] + horizonte < inicio_teste):
    raise RuntimeError("A amostra escolhida não pertence estritamente ao desenvolvimento.")

display(entrada_modelo.T.rename(columns={entrada_modelo.index[0]: "valor da amostra"}))

,valor da amostra
iqar_pm10,3.760000
iqar_pm2_5,6.406667
iqar_carbon_monoxide,0.728848
iqar_nitrogen_dioxide,1.540000
iqar_sulphur_dioxide,3.050000
iqar_ozone,25.200000
temperature_2m,19.800000
relative_humidity_2m,93.000000
precipitation,0.000000
wind_speed_10m,7.600000


## 4. Fazer a inferência

A probabilidade positiva é comparada ao limiar salvo no artefato. O texto “alerta” é apenas o resultado da regra experimental e não deve ser interpretado como aviso oficial.

In [5]:
if not hasattr(pipeline, "predict_proba"):
    raise TypeError("O pipeline empacotado não fornece predict_proba.")

probabilidade = float(pipeline.predict_proba(entrada_modelo)[0, 1])
classe_alerta = int(probabilidade >= limiar)
resultado = "ALERTA (regra experimental)" if classe_alerta else "Sem alerta pela regra"
horario = amostra["time"].iloc[0]
horario_evento = horario + horizonte

resumo = pd.DataFrame([{
    "horário dos dados até t": horario,
    "horário do evento previsto": horario_evento,
    "probabilidade estimada de IQAr > 100": probabilidade,
    "limiar congelado": limiar,
    "resultado da regra": resultado,
}])
display(resumo)

print(
    "Demonstração in-sample: este valor não é uma métrica de desempenho "
    "nem uma previsão atual."
)

,horário dos dados até t,horário do evento previsto,probabilidade estimada de IQAr > 100,limiar congelado,resultado da regra
0,2024-12-31 22:00:00,2024-12-31 23:00:00,0.0,0.3,Sem alerta pela regra


Demonstração in-sample: este valor não é uma métrica de desempenho nem uma previsão atual.
